# 01 - Data collection and exploration

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = '../data/raw/spotify_songs.csv'

## 1. Thu thập dữ liệu

### Chủ đề & Bối cảnh thực tế 
- Chủ đề: Metadata âm nhạc số và các đặc trưng âm thanh định lượng của 32.833 bài hát trên nền tảng Spotify, với 23 cột được chia thành 4 nhóm chính theo bảng mô tả của TidyTuesday:
  1. **Định danh bài hát & album (7 cột):** `track_id`, `track_name`, `track_artist`, `track_album_id`, `track_album_name`, `track_album_release_date`.
  2. **Phân loại theo playlist (4 cột):** `playlist_name`, `playlist_id`, `playlist_genre` (6 thể loại chính), `playlist_subgenre` (24 tiểu thể loại).
  3. **Chỉ số mức độ phổ biến (1 cột):** `track_popularity` (thang điểm 0–100 phản ánh độ thịnh hành của bài hát).
  4. **Đặc trưng âm thanh trích xuất từ Spotify API (12 cột):** `danceability`, `energy`, `key`, `loudness`, `mode`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, và `duration_ms`.
- Bối cảnh thực tế: Các nền tảng phát nhạc trực tuyến hiện đại như Spotify ứng dụng lĩnh vực Truy xuất Thông tin Âm nhạc (MIR) để lượng hóa cảm nhận âm nhạc trừu tượng thành các thông số kỹ thuật có cấu trúc. Bộ dữ liệu này phản ánh cách nền tảng dùng đặc trưng tín hiệu âm thanh kết hợp hành vi tuyển chọn playlist để vận hành hệ thống gợi ý và phân tích thị hiếu người nghe theo từng dòng nhạc.

### Nguồn dữ liệu
- **Bản gốc (TidyTuesday):** Dữ liệu từ **Spotify AB**, do **Kaylin Pavlik** thu thập và cộng đồng **R4DS** công bố ngày **21/01/2020** tại [https://github.com/rfordatascience/tidytuesday/tree/master/data/2020/2020-01-21](https://github.com/rfordatascience/tidytuesday/tree/master/data/2020/2020-01-21).
- **Bản tải về (Kaggle):** Được **Joakim Arvidsson** (`joebeachcapital`) đăng tải lại năm **2023** tại [https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs](https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs).

### Giấy phép sử dụng
- **Kho lưu trữ gốc (TidyTuesday):** Phát hành theo giấy phép **CC0-1.0** (Public Domain), cho phép tự do sao chép, chỉnh sửa và khai thác cho cả mục đích học thuật lẫn thương mại.
- **Bản phân phối (Kaggle):** Áp dụng **Open Database License (ODbL v1.0)**, yêu cầu ghi nhận nguồn đóng góp (R4DS, Kaylin Pavlik, Joakim Arvidsson) và duy trì tính mở nếu tái phân phối cơ sở dữ liệu phái sinh.
- **Ràng buộc bản quyền (Spotify Web API):** Dữ liệu gốc chịu sự điều chỉnh của **Spotify Developer Terms of Service**, giới hạn phạm vi sử dụng cho phân tích/nghiên cứu tổng hợp và cấm tái định danh người dùng cá nhân.

### Phương pháp thu thập & Thiên lệch dữ liệu
- **Cơ chế thu thập:** Truy xuất tự động qua **Spotify Web API** bằng gói thư viện `spotifyr` (R) từ các playlist công khai; tiêu chí chọn playlist được tác giả trình bày chi tiết trong bài blog đính kèm tại README gốc.
-  Lấy mẫu ~5.000 bài hát cho mỗi thể loại trong **6 thể loại chính** (*EDM, Latin, Pop, R&B, Rap, Rock* — chia thành 24 tiểu thể loại), tổng cộng **32.833 bản ghi**. Ngày phát hành trải dài từ **1957 đến 01/2020**, nhưng cấu trúc playlist và chỉ số `track_popularity` là ảnh chụp tĩnh (*cross-sectional snapshot*) tại **tháng 01/2020**.
- **Hạn chế & Biases:**
  - **Nhiễu nhãn thể loại (*Label Noise*):** Nhãn `playlist_genre` được kế thừa từ playlist chứa bài hát thay vì thể loại gốc của nghệ sĩ.
  - **Trùng lặp bản ghi (*Multi-playlist Duplication*):** Các bài hát phổ biến xuất hiện ở nhiều playlist gây trùng lặp `track_id` (4.477 bản ghi trùng).
  - **Thiên lệch chọn mẫu & văn hóa (*Sampling Bias*):** Tập trung vào 6 dòng nhạc thương mại phổ biến trên Spotify khiến dữ liệu nghiêng về nhạc phương Tây/tiếng Anh và loại trừ hoàn toàn các dòng nhạc ngoài danh mục và tiếng khác. (*Jazz, Classical, Folk...*).
  - **Thiên lệch thời gian (*Recency Bias*):** Điểm `track_popularity` (0–100) ưu tiên lượt phát gần thời điểm thu thập (01/2020), làm giảm đáng kể điểm số của các bản thu thập niên 1950–1990.
  - **Khuyết dữ liệu & Sai lệch định dạng:** Tồn tại 5 bản ghi khuyết (`NA`) ở thông tin định danh và sự không đồng nhất định dạng thời gian ở `track_album_release_date` (`YYYY` xen lẫn `YYYY-MM-DD`).
### Lý do chọn bộ dữ liệu
- **Cân bằng giữa quy mô và chiều sâu đặc trưng:** Với 32.833 quan sát cùng 23 thuộc tính kết hợp cả biến phân loại phân cấp (`playlist_genre`, `playlist_subgenre`), biến thời gian, biến mục tiêu (`track_popularity`) và 12 đặc trưng âm thanh liên tục, bộ dữ liệu đủ lớn và đa dạng để thực hiện trọn vẹn quy trình từ EDA, kiểm định thống kê đến xây dựng mô hình học máy.
- **Tính diễn giải trực quan:** Các đặc trưng như độ sôi động (`energy`), độ tích cực cảm xúc (`valence`), nhịp độ (`tempo`) hay độ ồn (`loudness`) dễ liên hệ thực tế, giúp nhóm diễn giải rõ ràng các phát hiện phân tích.
- **Cơ hội rèn luyện tiền xử lý thực tế:** Bộ dữ liệu chứa các vấn đề thực tế như trùng lặp `track_id` do giao thoa playlist, định dạng ngày phát hành không đồng nhất và phân phối lệch mạnh ở các biến như `instrumentalness`, `speechiness` — phù hợp để thể hiện kỹ thuật làm sạch và biến đổi dữ liệu.

In [5]:
df = pd.read_csv(DATA)
df.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,...,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


## 2. Data exploration

### 2.1 Overview

### 2.2 Numerical columns

### 2.3 Categorical columns

### 2.4 Missing data

### 2.5 Relationships

### 2.6 Summary
- Key observations
- Data quality issues
- Preprocessing plan
- Patterns that could lead to questions